In [0]:
%sql
USE CATALOG v_commerce;
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
catalog = "v_commerce"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# 1. Leitura da camada Bronze
tb_cliente_bronze = spark.table(f"{catalog}.{bronze_schema_name}.tb_clientes")

# 2. Removendo duplicados mantendo o cliente mais recente
window_cliente = Window.partitionBy("id_cliente").orderBy(
    F.to_date(F.col("data_cadastro")).desc_nulls_last()
)

tb_cliente_bronze = tb_cliente_bronze \
    .withColumn("rn", F.row_number().over(window_cliente)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

# 3. Limpeza e Padronização
tb_cliente_silver = tb_cliente_bronze.dropDuplicates(["id_cliente"]) \
    .withColumnRenamed("device_ids", "ids_dispositivos") \
    .withColumn("nome", F.initcap(F.col("nome"))) \
    .withColumn("sobrenome", F.initcap(F.col("sobrenome"))) \
    .withColumn("email", F.lower(F.col("email"))) \
    .withColumn("telefone", F.regexp_replace(F.col("telefone"), r"[^0-9]", "")) \
    .withColumn("data_nascimento", F.to_date(F.col("data_nascimento"))) \
    .withColumn("data_cadastro", F.to_date(F.col("data_cadastro"))) \
    .withColumn("genero", F.upper(F.col("genero"))) \
    .withColumn("origem", F.initcap(F.col("origem"))) \
    .fillna({"cidade": "Nao Informado", "estado": "Nao Informado"})

# 4. Criando colunas derivadas (Enriquecimento)
tb_cliente_silver = tb_cliente_silver \
    .withColumn("ano_cadastro", F.year(F.col("data_cadastro"))) \
    .withColumn("idade_aproximada", F.floor(F.datediff(F.current_date(), F.col("data_nascimento")) / 365.25))

# 5. Escrita na Camada Silver
tb_cliente_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{silver_schema_name}.tb_clientes")

In [0]:
#tb_cliente_silver.display()

In [0]:
# 1. Leitura da camada Bronze
tb_catalogo_produtos_bronze = spark.table(
    f"{catalog}.{bronze_schema_name}.tb_catalogo_produtos"
)

# 2. Removendo duplicados mantendo o produto mais recente
window_produto = Window.partitionBy("id_produto").orderBy(
    F.to_date(F.col("data_cadastro_produto")).desc_nulls_last()
)

tb_catalogo_produtos_bronze = tb_catalogo_produtos_bronze \
    .withColumn("rn", F.row_number().over(window_produto)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

# 3. Limpeza e Padronização
tb_catalogo_produtos_silver = tb_catalogo_produtos_bronze \
    .withColumn("nome_produto", F.initcap(F.trim(F.col("nome_produto")))) \
    .withColumn("fornecedor", F.initcap(F.trim(F.col("fornecedor")))) \
    .withColumn("data_cadastro_produto", F.to_date(F.col("data_cadastro_produto"))) \
    .withColumn("preco_limpo", F.regexp_replace(F.col("preco").cast("string"), r"[R\$\s]", "")) \
    .withColumn("preco_limpo", F.regexp_replace(F.col("preco_limpo"), ",", ".")) \
    .withColumn("preco_decimal", F.expr("try_cast(preco_limpo AS decimal(10,2))")) \
    .withColumn(
        "preco",
        F.when(
            (F.col("preco_decimal").isNull()) | (F.col("preco_decimal") <= 0),
            F.lit(-1).cast("decimal(10,2)")
        ).otherwise(F.col("preco_decimal"))
    ) \
    .withColumn("categoria_limpa", F.lower(F.trim(F.col("categoria")))) \
    .withColumn("categoria_limpa", F.regexp_replace("categoria_limpa", "0", "o")) \
    .withColumn("categoria_limpa", F.regexp_replace("categoria_limpa", "3", "e")) \
    .withColumn("categoria_limpa", F.regexp_replace("categoria_limpa", "1", "i")) \
    .withColumn("categoria_limpa", F.regexp_replace("categoria_limpa", "4", "a")) \
    .withColumn("categoria_limpa", F.regexp_replace("categoria_limpa", "5", "s")) \
    .withColumn("categoria_limpa", F.regexp_replace("categoria_limpa", r"[^a-zA-ZÀ-ÿ\s]", "")) \
    .withColumn(
        "categoria",
        F.when(
            F.col("categoria").isNull() |
            F.col("categoria_limpa").isin("", "null", "none", "nao informado", "não informado"),
            "Não informado"
        )
        .when(F.col("categoria_limpa").startswith("mov"), "Móveis")
        .when(F.col("categoria_limpa").startswith("brin"), "Brinquedos")
        .when(F.col("categoria_limpa").startswith("aut"), "Automotivo")
        .when(F.col("categoria_limpa").startswith("elet"), "Eletrônicos")
        .when(F.col("categoria_limpa").startswith("vest"), "Vestuário")
        .when(F.col("categoria_limpa").startswith("cas"), "Casa")
        .when(F.col("categoria_limpa").startswith("bel"), "Beleza")
        .when(F.col("categoria_limpa").startswith("esp"), "Esportes")
        .otherwise("Outros")
    ) \
    .withColumn(
        "ativo",
        F.when(
            F.lower(F.trim(F.col("ativo").cast("string"))).isin("sim", "s", "1", "true"),
            "Sim"
        )
        .when(
            F.lower(F.trim(F.col("ativo").cast("string"))).isin("nao", "não", "n", "0", "false"),
            "Nao"
        )
        .otherwise("Não informado")
    ) \
    .withColumn("peso_num", F.expr("try_cast(peso_kg AS double)")) \
    .withColumn(
        "peso_kg",
        F.when(
            (F.col("peso_num").isNull()) | (F.col("peso_num") <= 0),
            F.lit(-1.0)
        ).otherwise(F.col("peso_num"))
    ) \
    .withColumn(
        "estoque_disponivel",
        F.coalesce(
            F.expr("try_cast(estoque_disponivel AS int)"),
            F.lit(0)
        )
    ) \
    .drop("preco_limpo", "preco_decimal", "categoria_limpa", "peso_num")

# 4. Criando colunas derivadas
tb_catalogo_produtos_silver = tb_catalogo_produtos_silver \
    .withColumn("tem_estoque", F.when(F.col("estoque_disponivel") > 0, "Sim").otherwise("Nao")) \
    .withColumn(
        "precisa_revisao",
        F.when(
            (F.col("preco") == F.lit(-1).cast("decimal(10,2)")) |
            (F.col("peso_kg") == -1.0) |
            (F.col("categoria") == "Não informado") |
            (F.col("categoria") == "Outros") |
            (F.col("ativo") == "Não informado"),
            "Sim"
        ).otherwise("Nao")
    )

# 5. Escrita na Camada Silver
tb_catalogo_produtos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{silver_schema_name}.tb_catalogo_produtos")

In [0]:
#tb_catalogo_produtos_silver.display()